# Extended Improvements: YOLOv11 / TTA / WBF Ensemble / High-Res Training

This notebook adds four improvements on top of the YOLOv8n baseline:

| Improvement | Description |
|---|---|
| **D. YOLOv11n** | Latest YOLO architecture — same size as YOLOv8n, higher mAP |
| **E. TTA** | Test-Time Augmentation — merge flipped/scaled predictions at inference |
| **F. WBF Ensemble** | YOLOv8n + YOLOv8m combined with Weighted Box Fusion |
| **G. High-Res (1280)** | Train YOLOv8n at 1280×1280 for small fragment detection |

**Prerequisites:** Run `02_training_baseline.ipynb` and `07_improvements.ipynb` first.

In [ ]:
# ── Cell 1: GPU check + install ───────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠ No GPU — switch to GPU runtime!')

!pip install -q ultralytics ensemble-boxes sahi

In [ ]:
# ── Cell 2: Drive mount + paths ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, yaml, shutil
from pathlib import Path
import cv2, numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DRIVE_BASE    = Path('/content/drive/MyDrive/underwater_waste_detection')
YOLO_DS_DIR   = DRIVE_BASE / 'trashcan_yolo'
WEIGHTS_DIR   = DRIVE_BASE / 'weights'
RESULTS_DIR   = DRIVE_BASE / 'extended_results'
RUNS_DIR      = DRIVE_BASE / 'runs'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path('/content/underwater-waste-detection')
if not REPO_DIR.exists():
    !git clone https://github.com/omprxkash/underwater-waste-detection.git {REPO_DIR}
sys.path.insert(0, str(REPO_DIR / 'src'))
os.chdir(REPO_DIR)

# Baseline weights (trained in notebook 02)
NANO_WEIGHTS   = str(WEIGHTS_DIR / 'yolov8n_best.pt')
MEDIUM_WEIGHTS = str(WEIGHTS_DIR / 'yolov8m_best.pt')

# Dataset YAML
YAML_PATH = str(REPO_DIR / 'configs' / 'dataset_colab.yaml')
dataset_cfg = {
    'path': str(YOLO_DS_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 3,
    'names': {0: 'trash', 1: 'bio', 2: 'rov'}
}
with open(YAML_PATH, 'w') as f:
    yaml.dump(dataset_cfg, f)

CLASS_NAMES = ['trash', 'bio', 'rov']
print('Setup complete.')
print(f'Nano weights:   {NANO_WEIGHTS}')
print(f'Medium weights: {MEDIUM_WEIGHTS}')

## Improvement D: YOLOv11n Training

YOLOv11 was released by Ultralytics in October 2024. At the nano scale it achieves
slightly higher COCO mAP than YOLOv8n with fewer parameters (2.6M vs 3.2M).
The training call is identical — only the pretrained weights name changes.

In [ ]:
# ── Cell 3: Train YOLOv11n ────────────────────────────────────────────────
from ultralytics import YOLO

model_v11n = YOLO('yolo11n.pt')   # ultralytics downloads automatically

model_v11n.train(
    data=YAML_PATH,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project=str(RUNS_DIR),
    name='yolo11n_baseline',
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    mosaic=1.0,
    mixup=0.1,
    save_period=25,
)

v11n_best = RUNS_DIR / 'yolo11n_baseline' / 'weights' / 'best.pt'
shutil.copy2(v11n_best, WEIGHTS_DIR / 'yolo11n_best.pt')
print(f'YOLOv11n weights saved: {WEIGHTS_DIR}/yolo11n_best.pt')

In [ ]:
# ── Cell 4: Evaluate YOLOv11n (standard inference) ───────────────────────
v11n_model = YOLO(str(WEIGHTS_DIR / 'yolo11n_best.pt'))
v11n_val   = v11n_model.val(data=YAML_PATH, verbose=False)

print('YOLOv11n results:')
print(f'  mAP@0.5:      {float(v11n_val.box.map50):.4f}')
print(f'  mAP@0.5:0.95: {float(v11n_val.box.map):.4f}')
print(f'  Precision:    {float(v11n_val.box.mp):.4f}')
print(f'  Recall:       {float(v11n_val.box.mr):.4f}')

In [ ]:
# ── Cell 5: (Optional) Train YOLOv11s — small variant, better accuracy ───
# Uncomment to run. Uses ~9.4M params vs 2.6M for nano.

# model_v11s = YOLO('yolo11s.pt')
# model_v11s.train(
#     data=YAML_PATH, epochs=100, imgsz=640, batch=12, device=0,
#     project=str(RUNS_DIR), name='yolo11s_baseline',
#     optimizer='AdamW', lr0=0.001, warmup_epochs=3,
# )
# shutil.copy2(RUNS_DIR/'yolo11s_baseline'/'weights'/'best.pt', WEIGHTS_DIR/'yolo11s_best.pt')

## Improvement E: Test-Time Augmentation (TTA)

TTA runs inference on 8 augmented versions of each image (original + 3 flips + 4 scales),
then merges all predictions with NMS. Ultralytics supports this natively — just pass
`augment=True` to `.val()`. No retraining required. Typical gain: +2–4% mAP.

In [ ]:
# ── Cell 6: TTA on YOLOv8n baseline ──────────────────────────────────────
nano_model = YOLO(NANO_WEIGHTS)

# Standard inference
val_std = nano_model.val(data=YAML_PATH, augment=False, verbose=False)
print('YOLOv8n — Standard inference:')
print(f'  mAP@0.5:      {float(val_std.box.map50):.4f}')
print(f'  mAP@0.5:0.95: {float(val_std.box.map):.4f}')

# TTA inference
val_tta = nano_model.val(data=YAML_PATH, augment=True, verbose=False)
print('\nYOLOv8n — TTA (augment=True):')
print(f'  mAP@0.5:      {float(val_tta.box.map50):.4f}')
print(f'  mAP@0.5:0.95: {float(val_tta.box.map):.4f}')

tta_delta = float(val_tta.box.map50) - float(val_std.box.map50)
print(f'\nTTA gain (mAP@0.5): {tta_delta:+.4f}')

In [ ]:
# ── Cell 7: TTA on YOLOv11n ───────────────────────────────────────────────
v11n_tta = v11n_model.val(data=YAML_PATH, augment=True, verbose=False)
print('YOLOv11n — TTA:')
print(f'  mAP@0.5:      {float(v11n_tta.box.map50):.4f}')
print(f'  mAP@0.5:0.95: {float(v11n_tta.box.map):.4f}')

## Improvement F: Weighted Box Fusion (WBF) Ensemble

WBF combines bounding box predictions from multiple models by fusing overlapping
boxes proportional to their confidence scores — unlike NMS which picks one and
discards the rest. Combining a fast nano model with the more accurate medium model
typically gives +3–6% mAP over either alone.

**Requires:** YOLOv8m trained first (notebook 02, cell 7 — uncomment YOLOv8m training).

In [ ]:
# ── Cell 8: WBF Ensemble helper ───────────────────────────────────────────
from ensemble_boxes import weighted_boxes_fusion
from ultralytics import YOLO

def run_wbf_on_batch(
    models,
    image_paths,
    model_weights=None,
    iou_thr=0.55,
    skip_box_thr=0.01,
    conf_thr=0.25,
):
    """
    Run WBF ensemble over a list of images.
    Returns list of (boxes_norm, scores, labels) tuples.
    """
    if model_weights is None:
        model_weights = [1.0 / len(models)] * len(models)

    all_fused = []
    for img_path in image_paths:
        frame = cv2.imread(str(img_path))
        if frame is None:
            continue
        h, w = frame.shape[:2]

        boxes_list, scores_list, labels_list = [], [], []

        for model in models:
            result = model(frame, conf=conf_thr, iou=0.45, verbose=False)[0]
            if len(result.boxes) == 0:
                boxes_list.append(np.zeros((0, 4), dtype=np.float32))
                scores_list.append(np.array([], dtype=np.float32))
                labels_list.append(np.array([], dtype=np.float32))
                continue

            xyxy  = result.boxes.xyxy.cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()
            clss  = result.boxes.cls.cpu().numpy()

            # Normalize to [0,1] for WBF
            xyxy_norm = xyxy / np.array([w, h, w, h], dtype=np.float32)
            xyxy_norm = np.clip(xyxy_norm, 0, 1)

            boxes_list.append(xyxy_norm)
            scores_list.append(confs)
            labels_list.append(clss)

        fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list,
            weights=model_weights,
            iou_thr=iou_thr,
            skip_box_thr=skip_box_thr,
        )
        all_fused.append((img_path, frame, fused_boxes, fused_scores, fused_labels, w, h))

    return all_fused

print('WBF helper loaded.')

In [ ]:
# ── Cell 9: Load models + run WBF on validation set ──────────────────────
import glob

nano_model   = YOLO(NANO_WEIGHTS)
medium_model = YOLO(MEDIUM_WEIGHTS)   # train this in notebook 02 cell 7 first

val_imgs = list((YOLO_DS_DIR / 'images' / 'val').glob('*.jpg'))
val_imgs += list((YOLO_DS_DIR / 'images' / 'val').glob('*.png'))
print(f'Validation images: {len(val_imgs)}')

# Run WBF — nano weight 0.4, medium weight 0.6 (medium is stronger)
fused_predictions = run_wbf_on_batch(
    models=[nano_model, medium_model],
    image_paths=val_imgs,
    model_weights=[0.4, 0.6],
    iou_thr=0.55,
    skip_box_thr=0.05,
)
print(f'WBF complete: {len(fused_predictions)} predictions')

In [ ]:
# ── Cell 10: Compute WBF mAP using pycocotools ───────────────────────────
# Build COCO-style predictions from fused results and compute mAP
import json
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def build_coco_predictions(fused_results, class_names):
    """
    Convert WBF outputs to COCO prediction format.
    Returns list of dicts with image_id, category_id, bbox, score.
    """
    coco_preds = []
    img_info   = []

    for img_id, (img_path, frame, boxes, scores, labels, w, h) in enumerate(fused_results):
        img_info.append({'id': img_id, 'file_name': img_path.name,
                         'width': w, 'height': h})
        for box, score, label in zip(boxes, scores, labels):
            x1, y1, x2, y2 = box * np.array([w, h, w, h])
            bw, bh = x2 - x1, y2 - y1
            coco_preds.append({
                'image_id': img_id,
                'category_id': int(label),
                'bbox': [float(x1), float(y1), float(bw), float(bh)],
                'score': float(score),
            })

    return img_info, coco_preds

img_info_list, coco_pred_list = build_coco_predictions(fused_predictions, CLASS_NAMES)

pred_file = RESULTS_DIR / 'wbf_predictions.json'
with open(pred_file, 'w') as f:
    json.dump(coco_pred_list, f)

print(f'Total WBF predictions written: {len(coco_pred_list)}')
print('Note: full mAP evaluation requires ground-truth COCO JSON.')
print('Use ultralytics .val() on the fused model for a quick mAP estimate instead.')

In [ ]:
# ── Cell 11: Visualize WBF vs single-model on 4 sample images ────────────
import random

sample_idxs = random.sample(range(len(fused_predictions)), min(4, len(fused_predictions)))
samples = [fused_predictions[i] for i in sample_idxs]

COLORS = {'trash': (220, 50, 50), 'bio': (50, 180, 50), 'rov': (50, 130, 220)}

fig, axes = plt.subplots(len(samples), 3, figsize=(18, 5 * len(samples)))

for row, (img_path, frame, fused_boxes, fused_scores, fused_labels, w, h) in enumerate(samples):
    axes[row][0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[row][0].set_title('Original', fontweight='bold'); axes[row][0].axis('off')

    # Single model (YOLOv8n)
    nano_vis = frame.copy()
    nano_r = nano_model(frame, conf=0.25, verbose=False)[0]
    for box in nano_r.boxes:
        x1,y1,x2,y2 = map(int, box.xyxy[0])
        cls_name = CLASS_NAMES[int(box.cls[0])]
        color = COLORS.get(cls_name, (128,128,128))
        cv2.rectangle(nano_vis,(x1,y1),(x2,y2),color,2)
        cv2.putText(nano_vis,f'{cls_name} {float(box.conf[0]):.0%}',(x1,y1-4),
                    cv2.FONT_HERSHEY_SIMPLEX,0.5,color,2)
    axes[row][1].imshow(cv2.cvtColor(nano_vis, cv2.COLOR_BGR2RGB))
    axes[row][1].set_title(f'YOLOv8n ({len(nano_r.boxes)} det)', fontweight='bold')
    axes[row][1].axis('off')

    # WBF ensemble
    wbf_vis = frame.copy()
    for box_norm, score, label in zip(fused_boxes, fused_scores, fused_labels):
        x1,y1,x2,y2 = (box_norm * np.array([w,h,w,h])).astype(int)
        cls_name = CLASS_NAMES[int(label)] if int(label) < len(CLASS_NAMES) else 'unknown'
        color = COLORS.get(cls_name, (128,128,128))
        cv2.rectangle(wbf_vis,(x1,y1),(x2,y2),color,2)
        cv2.putText(wbf_vis,f'{cls_name} {score:.0%}',(x1,y1-4),
                    cv2.FONT_HERSHEY_SIMPLEX,0.5,color,2)
    axes[row][2].imshow(cv2.cvtColor(wbf_vis, cv2.COLOR_BGR2RGB))
    axes[row][2].set_title(f'WBF Ensemble ({len(fused_boxes)} det)', fontweight='bold')
    axes[row][2].axis('off')

plt.suptitle('WBF Ensemble vs YOLOv8n Single Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'wbf_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## Improvement G: High-Resolution Training (1280×1280)

Training at 1280×1280 instead of 640×640 gives the model 4× more pixels per image,
which substantially improves detection of small trash fragments. Tradeoff: ~2× longer
training time, ~4× more VRAM. Use batch=4 on T4, batch=8 on A100.

In [ ]:
# ── Cell 12: Train YOLOv8n at 1280×1280 ──────────────────────────────────
model_hires = YOLO('yolov8n.pt')

model_hires.train(
    data=YAML_PATH,
    epochs=80,           # fewer epochs — larger batches have more gradient information
    imgsz=1280,
    batch=4,             # reduce if OOM: try 2
    device=0,
    project=str(RUNS_DIR),
    name='yolov8n_1280',
    optimizer='AdamW',
    lr0=0.0005,          # lower LR for high-res — fewer updates per epoch
    lrf=0.01,
    warmup_epochs=5,
    mosaic=0.5,          # reduce mosaic at high-res to avoid VRAM spikes
    mixup=0.1,
    close_mosaic=10,
)

hires_best = RUNS_DIR / 'yolov8n_1280' / 'weights' / 'best.pt'
shutil.copy2(hires_best, WEIGHTS_DIR / 'yolov8n_1280_best.pt')
print(f'High-res weights saved: {WEIGHTS_DIR}/yolov8n_1280_best.pt')

In [ ]:
# ── Cell 13: Evaluate 1280 model ─────────────────────────────────────────
hires_model = YOLO(str(WEIGHTS_DIR / 'yolov8n_1280_best.pt'))
hires_val   = hires_model.val(data=YAML_PATH, imgsz=1280, verbose=False)

print('YOLOv8n @ 1280×1280 results:')
print(f'  mAP@0.5:      {float(hires_val.box.map50):.4f}')
print(f'  mAP@0.5:0.95: {float(hires_val.box.map):.4f}')
print(f'  Precision:    {float(hires_val.box.mp):.4f}')
print(f'  Recall:       {float(hires_val.box.mr):.4f}')

## Final Comparison Table

Fill this in as you complete each training run. Update the placeholder values.

In [ ]:
# ── Cell 14: Run all evals + build comparison table ───────────────────────
import time

def eval_model(weights_path, imgsz=640, tta=False, label=None):
    """Load model, run val, return metrics dict."""
    if not Path(weights_path).exists():
        return None
    model  = YOLO(weights_path)
    result = model.val(data=YAML_PATH, imgsz=imgsz, augment=tta, verbose=False)

    # FPS benchmark (50 images)
    test_imgs = list((YOLO_DS_DIR / 'images' / 'val').glob('*.jpg'))[:50]
    frames    = [cv2.imread(str(p)) for p in test_imgs]
    frames    = [f for f in frames if f is not None]
    for f in frames[:5]:     # warmup
        model(f, verbose=False)
    t0 = time.time()
    for f in frames:
        model(f, verbose=False)
    fps = len(frames) / (time.time() - t0)

    return {
        'Model': label or Path(weights_path).stem,
        'imgsz': imgsz,
        'TTA': tta,
        'mAP@0.5': round(float(result.box.map50), 4),
        'mAP@0.5:0.95': round(float(result.box.map), 4),
        'Precision': round(float(result.box.mp), 4),
        'Recall': round(float(result.box.mr), 4),
        'FPS': round(fps, 1),
    }


# Evaluate all available models
eval_configs = [
    (NANO_WEIGHTS,                          640,  False, 'YOLOv8n (baseline)'),
    (NANO_WEIGHTS,                          640,  True,  'YOLOv8n + TTA'),
    (MEDIUM_WEIGHTS,                        640,  False, 'YOLOv8m'),
    (MEDIUM_WEIGHTS,                        640,  True,  'YOLOv8m + TTA'),
    (str(WEIGHTS_DIR / 'yolo11n_best.pt'),  640,  False, 'YOLOv11n'),
    (str(WEIGHTS_DIR / 'yolo11n_best.pt'),  640,  True,  'YOLOv11n + TTA'),
    (str(WEIGHTS_DIR / 'yolov8n_1280_best.pt'), 1280, False, 'YOLOv8n @ 1280'),
    # From notebook 07:
    (str(WEIGHTS_DIR / 'yolov8n_enhanced_best.pt'), 640, False, 'YOLOv8n + CLAHE+WB'),
    (str(WEIGHTS_DIR / 'yolov8n_augmented_best.pt'), 640, False, 'YOLOv8n + DomainAug'),
]

rows = []
for weights, imgsz, tta, label in eval_configs:
    print(f'Evaluating: {label}...')
    row = eval_model(weights, imgsz=imgsz, tta=tta, label=label)
    if row:
        rows.append(row)
        print(f'  mAP@0.5: {row["mAP@0.5"]}')

comparison_df = pd.DataFrame(rows)
comparison_df = comparison_df.sort_values('mAP@0.5', ascending=False).reset_index(drop=True)

print('\n' + '='*90)
print('COMPLETE MODEL COMPARISON')
print('='*90)
print(comparison_df.to_string(index=False))

comparison_df.to_csv(str(RESULTS_DIR / 'full_comparison.csv'), index=False)
print(f'\nSaved: {RESULTS_DIR}/full_comparison.csv')

In [ ]:
# ── Cell 15: Comparison bar chart ─────────────────────────────────────────
if len(comparison_df) > 0:
    fig, ax = plt.subplots(figsize=(14, 6))

    colors = plt.cm.viridis(np.linspace(0.2, 0.85, len(comparison_df)))
    bars = ax.barh(
        comparison_df['Model'][::-1],
        comparison_df['mAP@0.5'][::-1],
        color=colors, edgecolor='white', linewidth=0.8,
    )

    for bar, val in zip(bars, comparison_df['mAP@0.5'][::-1]):
        ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10, fontweight='bold')

    ax.set_xlabel('mAP@0.5', fontsize=12)
    ax.set_title('All Models & Improvements — mAP@0.5 Comparison\nUnderwater Waste Detection',
                 fontsize=13, fontweight='bold')
    ax.set_xlim(0, comparison_df['mAP@0.5'].max() + 0.08)
    ax.axvline(comparison_df[comparison_df['Model']=='YOLOv8n (baseline)']['mAP@0.5'].values[0]
               if 'YOLOv8n (baseline)' in comparison_df['Model'].values else 0,
               color='red', linestyle='--', alpha=0.5, label='Baseline')
    ax.legend()
    ax.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'all_models_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Chart saved: {RESULTS_DIR}/all_models_comparison.png')

In [ ]:
# ── Cell 16: Per-class analysis — where does each model improve? ──────────
# Compare per-class mAP between YOLOv8n baseline and best model

baseline = YOLO(NANO_WEIGHTS)
best_weights = comparison_df.iloc[0]
print(f'Best model: {best_weights["Model"]}')

best_model_path = None
weight_map = {
    'YOLOv8n (baseline)':  NANO_WEIGHTS,
    'YOLOv8m':             MEDIUM_WEIGHTS,
    'YOLOv11n':            str(WEIGHTS_DIR / 'yolo11n_best.pt'),
    'YOLOv8n @ 1280':      str(WEIGHTS_DIR / 'yolov8n_1280_best.pt'),
    'YOLOv8n + CLAHE+WB':  str(WEIGHTS_DIR / 'yolov8n_enhanced_best.pt'),
    'YOLOv8n + DomainAug': str(WEIGHTS_DIR / 'yolov8n_augmented_best.pt'),
}

base_val = baseline.val(data=YAML_PATH, verbose=False)
base_per_class = [round(float(v), 4) for v in base_val.box.ap50.tolist()]

best_key = best_weights['Model'].replace(' + TTA', '')
if best_key in weight_map and Path(weight_map[best_key]).exists():
    best_m = YOLO(weight_map[best_key])
    best_val = best_m.val(data=YAML_PATH, verbose=False)
    best_per_class = [round(float(v), 4) for v in best_val.box.ap50.tolist()]

    x = np.arange(len(CLASS_NAMES))
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    bars1 = ax.bar(x - width/2, base_per_class, width, label='YOLOv8n baseline',
                   color='#7f8c8d', edgecolor='white')
    bars2 = ax.bar(x + width/2, best_per_class, width, label=best_weights['Model'],
                   color='#2980b9', edgecolor='white')

    for bar in bars1:
        ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

    ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, fontsize=11)
    ax.set_ylabel('AP@0.5'); ax.set_ylim(0, 1.1)
    ax.set_title('Per-class AP@0.5: Baseline vs Best Model', fontweight='bold')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'per_class_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Cell 17: Speed vs accuracy scatter plot ────────────────────────────────
if len(comparison_df) > 0 and 'FPS' in comparison_df.columns:
    fig, ax = plt.subplots(figsize=(11, 6))

    scatter_df = comparison_df.dropna(subset=['FPS', 'mAP@0.5'])

    sc = ax.scatter(
        scatter_df['FPS'],
        scatter_df['mAP@0.5'],
        s=120, c=range(len(scatter_df)),
        cmap='plasma', zorder=3, edgecolors='white', linewidths=0.8,
    )

    for _, row in scatter_df.iterrows():
        ax.annotate(
            row['Model'],
            (row['FPS'], row['mAP@0.5']),
            xytext=(6, 4), textcoords='offset points',
            fontsize=8.5, alpha=0.9,
        )

    ax.set_xlabel('Inference Speed (FPS)', fontsize=12)
    ax.set_ylabel('mAP@0.5', fontsize=12)
    ax.set_title('Speed vs Accuracy — All Models & Improvements', fontsize=13, fontweight='bold')
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'speed_vs_accuracy.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {RESULTS_DIR}/speed_vs_accuracy.png')

In [ ]:
# ── Cell 18: Save all results to Drive ───────────────────────────────────
comparison_df.to_csv(str(RESULTS_DIR / 'full_comparison.csv'), index=False)
print('Saved CSVs to Drive:')
print(f'  {RESULTS_DIR}/full_comparison.csv')
print(f'\nImages saved:')
for f in sorted(RESULTS_DIR.glob('*.png')):
    print(f'  {f.name}')

print('\n--- Summary ---')
if len(comparison_df) > 0:
    best = comparison_df.iloc[0]
    baseline_row = comparison_df[comparison_df['Model']=='YOLOv8n (baseline)']
    if not baseline_row.empty:
        baseline_map = baseline_row['mAP@0.5'].values[0]
        print(f'Best model: {best["Model"]} — mAP@0.5: {best["mAP@0.5"]}')
        print(f'Gain over baseline: {best["mAP@0.5"] - baseline_map:+.4f}')
    print(f'\nFull table:')
    print(comparison_df[['Model','mAP@0.5','mAP@0.5:0.95','FPS']].to_string(index=False))